## imports

### reload

In [61]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### imports dependencies

In [ ]:
import json
import sys
import os
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

# ingestion
from app.ingestion.pipeline import chunk_ingestion_pipeline, question_ingestion_pipeline
from app.schemas import DocumentEval

# rag 
from app.rag.pipeline import rag_pipeline

# storage 
from app.storage.setup import setup_chunk_storage, setup_question_storage, delete_all_docs

# input_filtering
from app.input_filtering.filtering import filter

# langchain
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings

# dotenv 
from dotenv import load_dotenv

# visualization
from tqdm import tqdm

# util
from app.util.time_util import current_timestamp_str

# config 
from application.config import EMBEDDING_MODEL

In [ ]:
with open("../dataset/machine_learning_knowledge.json") as f:
    json_docs = json.load(f)
    print(f"docs: {json_docs[0]}")

with open("../dataset/machine_learning_knowledge_question_doc_mapping.json") as f:
    json_question_doc_mapping = json.load(f)
    print(f"docs_question: {json_question_doc_mapping[0]}")

docs: {'id': 'doc1', 'text': 'Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed. Algorithms analyze historical data and use it to make predictions or decisions. It is widely used in applications such as recommendation systems, fraud detection, and image recognition. The performance of a machine learning model typically improves as it is exposed to more high-quality data.'}
docs_question: {'id': 'q1', 'question': 'What does machine learning mean and what is it used for?', 'docs': ['doc1', 'doc2', 'doc4', 'doc9']}


In [64]:
doc_source = [Document(metadata={"id": doc["id"]}, page_content=doc["text"]) for doc in json_docs]
doc_question = [Document(metadata={"docs": json.dumps(doc["docs"]), "id":doc["id"]}, page_content=doc["question"]) for doc in json_question_doc_mapping]

print(doc_source)
print(doc_question)

[Document(metadata={'id': 'doc1'}, page_content='Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed. Algorithms analyze historical data and use it to make predictions or decisions. It is widely used in applications such as recommendation systems, fraud detection, and image recognition. The performance of a machine learning model typically improves as it is exposed to more high-quality data.'), Document(metadata={'id': 'doc2'}, page_content='Supervised learning is a type of machine learning where models are trained using labeled data. Each training example contains both input features and the correct output. The algorithm learns a mapping from inputs to outputs so it can make predictions on new data. Common supervised tasks include classification and regression.'), Document(metadata={'id': 'doc3'}, page_content='Unsupervised learning involves training models on data that does not conta

### setup storage

In [65]:
embedding_function = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore_chunk = setup_chunk_storage(embedding_function)
vectorstore_question = setup_question_storage(embedding_function)

delete_all_docs(vectorstore_chunk)
delete_all_docs(vectorstore_question)

Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### setup dotenv

In [66]:
load_dotenv(dotenv_path='../.env')

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## ingest

In [ ]:
#TODO: sanitize input data before ingestion (e.g. dangerous prompt, PII, etc.)

question_ingestion_pipeline(vectorstore_question, doc_question)
chunk_ingestion_pipeline(vectorstore_chunk, doc_source)

doc_question size: 12
collection size: 12
doc_source size: 20; chunked_docs size: 43
collection size: 43


[Document(metadata={'id': 'doc1', 'chunk_id': 'doc1_chunk0'}, page_content='Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed'),
 Document(metadata={'id': 'doc1', 'chunk_id': 'doc1_chunk1'}, page_content='. Algorithms analyze historical data and use it to make predictions or decisions. It is widely used in applications such as recommendation systems, fraud detection, and image recognition'),
 Document(metadata={'id': 'doc1', 'chunk_id': 'doc1_chunk2'}, page_content='. The performance of a machine learning model typically improves as it is exposed to more high-quality data.'),
 Document(metadata={'id': 'doc2', 'chunk_id': 'doc2_chunk0'}, page_content='Supervised learning is a type of machine learning where models are trained using labeled data. Each training example contains both input features and the correct output'),
 Document(metadata={'id': 'doc2', 'chunk_id': 'doc2_chunk1'}, pag

## rag pipeline

In [68]:
question_before_filtering = "What is the main goal of unsupervised learning?"
valid, question = filter(question_before_filtering)

print(f"before filtering: {question_before_filtering}")
print(f"after filtering: {question}")

before filtering: What is the main goal of unsupervised learning?
after filtering: What is the main goal of unsupervised learning?


In [71]:
answer, contexts, system_prompt, user_prompt = rag_pipeline(vectorstore_chunk, vectorstore_question, question)
print(f"system_prompt: {system_prompt}")
print("-----")
print(f"user_prompt: {user_prompt}")

question: What is the main goal of unsupervised learning?
expanded_question: ['Clustering', 'Dimensionality Reduction', 'Anomaly Detection', 'What is the main goal of unsupervised learning?']
context_result: 5
unique context len: 4
before rerank: 
[Document(metadata={'chunk_id': 'doc10_chunk0', 'id': 'doc10', 'distance': 0.1793166995048523}, page_content='Clustering is an unsupervised learning technique used to group similar data points together. Unlike classification, clustering does not rely on labeled data'), Document(metadata={'chunk_id': 'doc10_chunk1', 'id': 'doc10', 'distance': 0.36656874418258667}, page_content='Algorithms such as k-means and hierarchical clustering are commonly used. Clustering is often applied in customer segmentation and anomaly detection.'), Document(metadata={'chunk_id': 'doc3_chunk1', 'id': 'doc3', 'distance': 0.4967091679573059}, page_content='Techniques such as clustering and dimensionality reduction are commonly used. These methods are helpful for expl

In [72]:
print("===== ANSWER =====")
print(answer)
print("--------")
print("===== CONTEXT =====")
print(contexts)

===== ANSWER =====
The main goal of unsupervised learning is to discover hidden patterns or structures within the dataset, typically without labeled data.
--------
===== CONTEXT =====
[Document(metadata={'chunk_id': 'doc3_chunk0', 'id': 'doc3', 'distance': 0.28105175495147705}, page_content='Unsupervised learning involves training models on data that does not contain labels. The goal is to discover hidden patterns or structures within the dataset'), Document(metadata={'chunk_id': 'doc10_chunk0', 'id': 'doc10', 'distance': 0.1793166995048523}, page_content='Clustering is an unsupervised learning technique used to group similar data points together. Unlike classification, clustering does not rely on labeled data'), Document(metadata={'chunk_id': 'doc3_chunk1', 'id': 'doc3', 'distance': 0.4967091679573059}, page_content='Techniques such as clustering and dimensionality reduction are commonly used. These methods are helpful for exploratory data analysis and feature discovery.'), Document(m

In [73]:
# search_result = vectorstore.similarity_search_with_score(question, k=5)
# print(search_result)

## evaluation

In [ ]:
with open("../dataset/machine_learning_knowledge_evaluation.json") as f:
    doc_eval_json = json.load(f)
    print(doc_eval_json)

[{'question': 'What is machine learning and what is its main goal?', 'answer': 'Machine learning is a field of artificial intelligence that enables computers to learn patterns from data and make predictions or decisions without explicit programming.', 'relevant_doc_ids': ['doc1']}, {'question': 'Give two examples of applications where machine learning is commonly used.', 'answer': 'Machine learning is commonly used in applications such as recommendation systems, fraud detection, and image recognition.', 'relevant_doc_ids': ['doc1']}, {'question': 'What type of machine learning uses labeled data for training?', 'answer': 'Supervised learning uses labeled data where each training example includes input features and the correct output.', 'relevant_doc_ids': ['doc2']}, {'question': 'What are two common tasks in supervised learning?', 'answer': 'Two common supervised learning tasks are classification and regression.', 'relevant_doc_ids': ['doc2', 'doc8', 'doc9']}, {'question': 'What is the 

In [ ]:
doc_eval = [DocumentEval(
                question=doc['question'], 
                answer=doc['answer'], 
                relevant_doc_ids=doc['relevant_doc_ids']) 
            for doc in doc_eval_json]

In [ ]:
for doc in doc_eval[:3]:
    print(doc.question)
    print(doc.answer)
    print(doc.relevant_doc_ids)
    print("----------")

What is machine learning and what is its main goal?
Machine learning is a field of artificial intelligence that enables computers to learn patterns from data and make predictions or decisions without explicit programming.
['doc1']
----------
Give two examples of applications where machine learning is commonly used.
Machine learning is commonly used in applications such as recommendation systems, fraud detection, and image recognition.
['doc1']
----------
What type of machine learning uses labeled data for training?
Supervised learning uses labeled data where each training example includes input features and the correct output.
['doc2']
----------


### run all cases

In [ ]:
answers = []
user_prompts = []
retrieved_contexts = []

for doc in tqdm(doc_eval): 
    answer, contexts, _, user_prompt = rag_pipeline(vectorstore_chunk, vectorstore_question, doc.question)
    if answer is None:
        answers.append("I don't know")
    else:
        answers.append(answer)

    user_prompts.append(user_prompt)
    retrieved_contexts.append(contexts)

100%|██████████| 27/27 [16:57<00:00, 37.69s/it]


In [ ]:
retrieved_contexts[1]

[Document(metadata={'chunk_id': 'doc2_chunk0', 'id': 'doc2', 'distance': 0.4769992530345917}, page_content='Supervised learning is a type of machine learning where models are trained using labeled data. Each training example contains both input features and the correct output'),
 Document(metadata={'id': 'doc1', 'chunk_id': 'doc1_chunk0', 'distance': 0.37436601519584656}, page_content='Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed'),
 Document(metadata={'id': 'doc1', 'chunk_id': 'doc1_chunk1', 'distance': 0.4556378424167633}, page_content='Algorithms analyze historical data and use it to make predictions or decisions. It is widely used in applications such as recommendation systems, fraud detection, and image recognition'),
 Document(metadata={'chunk_id': 'doc17_chunk1', 'id': 'doc17', 'distance': 0.43931350111961365}, page_content='Examples include learning rate, number of layer

### recall@k

In [ ]:
k = 5 

total = 0.0
valid_cases = 0

def recall_at_k(relevant, retrieved, k):
    if not relevant:
        return None, set()

    if not retrieved:
        return 0.0, set()

    retrieved_k = retrieved[:k]

    relevant_set = set(relevant)
    retrieved_set = set(retrieved_k)

    relevant_found = relevant_set.intersection(retrieved_set)

    recall = len(relevant_found) / len(relevant_set)

    return recall, relevant_found

for answer, retrieved, eval in tqdm(zip(answers, retrieved_contexts, doc_eval)):
    relevant = eval.relevant_doc_ids

    print(f"answer\t\t: {answer}")
    print(f"relevant\t: {relevant}")

    # get retrieved
    if retrieved is None:
        score = 0.0
        print("retrieved\t: None")
    else:
        retrieved_doc_ids = [r.metadata['id'] for r in retrieved]
        retrieved_doc_ids = list(dict.fromkeys(retrieved_doc_ids))
        print(f"retrieved_doc_ids: {retrieved_doc_ids}")

        score, relevant_found = recall_at_k(relevant, retrieved_doc_ids, k)
    
    print(f"recall@{k} score\t: {score}")
    print("=========================================")

    if score is not None:
        total += score
        valid_cases += 1


average = total / valid_cases if valid_cases > 0 else 0
print(f"average score: {average:.2f}")

27it [00:00, 13895.24it/s]

answer		: Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed. It involves the optimization of computer algorithms for tasks such as classification, regression, pattern recognition, natural language processing, speech recognition, image analysis, speech-to-text synthesis, and knowledge-based systems. The main goal of machine learning is to automate these processes so that machines can perform tasks that were previously done by humans with fewer human errors.
relevant	: ['doc1']
retrieved_doc_ids: ['doc1', 'doc13', 'doc2', 'doc9']
recall@5 score	: 1.0
answer		: Example 1: Machine learning is often used in recommendation systems to suggest products or services to customers based on their previous behavior, preferences, or demographic information. 
Example 2: Machine learning can be used for fraud detection by identifying patterns that indicate an unusual transaction or change in behavior

In [ ]:
print(f"average score: {average:.2f}")

average score: 0.93


### generation evaluation - ragas

In [ ]:
questions = [eval.question for eval in doc_eval]
ground_truths = [eval.answer for eval in doc_eval]

relevant_doc_ids   = [r.relevant_doc_ids for r in doc_eval]

contexts = []
for doc_list in retrieved_contexts:
    if doc_list is None:
        contexts.append([])
    else:
        contexts.append([doc.page_content for doc in doc_list])

print(contexts)
print(ground_truths)


[['Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed', 'Gradient descent is an optimization algorithm used to train many machine learning models. It works by iteratively adjusting model parameters to minimize a loss function', 'The algorithm learns a mapping from inputs to outputs so it can make predictions on new data. Common supervised tasks include classification and regression.', 'Popular algorithms include logistic regression, support vector machines, and random forests. Accuracy, precision, recall, and F1-score are common evaluation metrics.'], ['Supervised learning is a type of machine learning where models are trained using labeled data. Each training example contains both input features and the correct output', 'Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed', 'Algorith

In [ ]:
from datasets import Dataset

print(f"question len    : {len(questions)}")
print(f"contexts len    : {len(contexts)}")
print(f"response len    : {len(answers)}")
print(f"ground_truth len: {len(ground_truths)}")

data = {
    "question": questions,
    "contexts": contexts,
    "answer": answers,
    "ground_truth": ground_truths
}

dataset = Dataset.from_dict(data)

question len    : 27
contexts len    : 27
response len    : 27
ground_truth len: 27


In [ ]:
# dump all dataset
filename = f"ragas_dataset_{current_timestamp_str()}.json"
with open(filename, "w") as f:
    json.dump(data, f)
print(filename)

ragas_dataset_20260327_162118.json


In [ ]:
# import sys
# sys.exit()

In [ ]:
print('questions')
[print(q) for q in questions[:3]]
print('='*10)
print('answers')
[print(a+"\n-----") for a in answers[:3]]
print('='*10)
print('prompts')
[print(p+"\n-----") for p in user_prompts[:3] if p is not None]

questions
What is machine learning and what is its main goal?
Give two examples of applications where machine learning is commonly used.
What type of machine learning uses labeled data for training?
answers
Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed. It involves the optimization of computer algorithms for tasks such as classification, regression, pattern recognition, natural language processing, speech recognition, image analysis, speech-to-text synthesis, and knowledge-based systems. The main goal of machine learning is to automate these processes so that machines can perform tasks that were previously done by humans with fewer human errors.
-----
Example 1: Machine learning is often used in recommendation systems to suggest products or services to customers based on their previous behavior, preferences, or demographic information. 
Example 2: Machine learning can be used for

[None, None, None]

In [ ]:
retrieved_contexts[1]

[Document(metadata={'chunk_id': 'doc2_chunk0', 'id': 'doc2', 'distance': 0.4769992530345917}, page_content='Supervised learning is a type of machine learning where models are trained using labeled data. Each training example contains both input features and the correct output'),
 Document(metadata={'id': 'doc1', 'chunk_id': 'doc1_chunk0', 'distance': 0.37436601519584656}, page_content='Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed'),
 Document(metadata={'id': 'doc1', 'chunk_id': 'doc1_chunk1', 'distance': 0.4556378424167633}, page_content='Algorithms analyze historical data and use it to make predictions or decisions. It is widely used in applications such as recommendation systems, fraud detection, and image recognition'),
 Document(metadata={'chunk_id': 'doc17_chunk1', 'id': 'doc17', 'distance': 0.43931350111961365}, page_content='Examples include learning rate, number of layer

In [ ]:
print(user_prompts[0])


Context:
- Machine learning is a field of artificial intelligence that focuses on enabling computers to learn patterns from data without being explicitly programmed
- Gradient descent is an optimization algorithm used to train many machine learning models. It works by iteratively adjusting model parameters to minimize a loss function
- The algorithm learns a mapping from inputs to outputs so it can make predictions on new data. Common supervised tasks include classification and regression.
- Popular algorithms include logistic regression, support vector machines, and random forests. Accuracy, precision, recall, and F1-score are common evaluation metrics.


Question:
What is machine learning and what is its main goal?



In [ ]:
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from application.config import EVALUATION_MODEL

"""
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    temperature=0,
    model_kwargs={"n": 1}
)
"""

llm = ChatOpenAI(
    model=EVALUATION_MODEL,
    api_key=OPENAI_API_KEY,
    temperature=0,
    model_kwargs={"n": 1},
    timeout=600,
    max_retries=10
)

ragas_llm = LangchainLLMWrapper(llm)

d:\Programs\anaconda3\envs\rag_py3.10\lib\site-packages\IPython\core\interactiveshell.py:3519: UserWarning: Parameters {'n'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):
C:\Users\april\AppData\Local\Temp\ipykernel_6248\3792604272.py:24: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)


In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_precision,
    context_recall
)

result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        # answer_relevancy,
        answer_correctness,
        # context_precision,
        context_recall
    ],
    llm=ragas_llm,
    embeddings=embedding_function
)

C:\Users\april\AppData\Local\Temp\ipykernel_6248\2749274894.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\april\AppData\Local\Temp\ipykernel_6248\2749274894.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\april\AppData\Local\Temp\ipykernel_6248\2749274894.py:2: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import (
C:\Users\april\AppData\Local\Temp\ipykernel_624

Evaluating:   0%|          | 0/81 [00:00<?, ?it/s]

In [ ]:
print(result)
  
df = result.to_pandas()
df.head()

{'faithfulness': 0.6918, 'answer_correctness': 0.6094, 'context_recall': 1.0000}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_correctness,context_recall
0,What is machine learning and what is its main ...,[Machine learning is a field of artificial int...,Machine learning is a field of artificial inte...,Machine learning is a field of artificial inte...,0.285714,0.570152,1.0
1,Give two examples of applications where machin...,[Supervised learning is a type of machine lear...,Example 1: Machine learning is often used in r...,Machine learning is commonly used in applicati...,0.500000,0.977878,1.0
2,What type of machine learning uses labeled dat...,[Supervised learning is a type of machine lear...,Supervised learning,Supervised learning uses labeled data where ea...,1.000000,0.711820,1.0
3,What are two common tasks in supervised learning?,[The algorithm learns a mapping from inputs to...,Classification and regression,Two common supervised learning tasks are class...,1.000000,0.972012,1.0
4,What is the main goal of unsupervised learning?,[Unsupervised learning involves training model...,The goal is to discover hidden patterns or str...,The goal of unsupervised learning is to discov...,1.000000,0.971599,1.0


In [ ]:
filename = f'result_{current_timestamp_str()}.xlsx'
df.to_excel(filename)
print(filename)

result_20260327_162524.xlsx


In [ ]:
print(system_prompt)


Answer the question using only the context.

Instructions:
- Use exact words and phrases from the context
- Copy the answer directly from the context whenever possible
- Select the shortest span that fully answers the question
- If multiple contexts are provided, choose the context that contains the most complete answer

Constraints:
- Use only information from the context
- Keep the answer to one sentence

If there is no relevant answer in the contexts, say:
I don't know



In [ ]:
print(user_prompts[9])


Context:
- Techniques such as regularization, dropout, and cross-validation help reduce overfitting. A good model balances learning patterns while maintaining generalization.
- Regularization is a technique used to prevent machine learning models from overfitting. It adds a penalty term to the loss function that discourages overly complex models
- Increasing model complexity or adding better features may help solve this problem. Proper model selection is important to avoid underfitting.
- Overfitting occurs when a machine learning model learns the training data too well, including its noise and irrelevant details. As a result, the model performs poorly on new, unseen data
- Common regularization methods include L1 and L2 regularization. These techniques help improve the model's ability to generalize.


Question:
Name two techniques that can help reduce overfitting.



## Sandbox

In [ ]:
answers[6]

'The context does not provide any information about the three common dataset splits used in machine learning. However, I would suggest looking at a practical example or further context to answer this question.'

In [ ]:
user_prompts[6]

'\nContext:\n- A dataset in machine learning is typically divided into training, validation, and test sets. The training set is used to teach the model patterns in the data\n- Cross-validation is a method used to evaluate machine learning models more reliably. The dataset is divided into multiple folds, and the model is trained and tested several times\n- Model evaluation is the process of measuring how well a machine learning model performs on unseen data. Different tasks use different evaluation metrics\n- Each fold serves as a test set once while the others are used for training. This approach provides a more robust estimate of model performance.\n- Underfitting happens when a model is too simple to capture the underlying structure of the data. In this case, the model performs poorly on both training and test datasets\n\n\nQuestion:\nWhat are the three common dataset splits used in machine learning?\n'

In [ ]:
print(question)

What is the main goal of unsupervised learning?


In [ ]:
from app.rag.pipeline import get_context_based_on_question, get_context_based_on_question_mapping
context_result = get_context_based_on_question(vectorstore_chunk, question)
context_result_question, q_results = get_context_based_on_question_mapping(vectorstore_chunk, vectorstore_question, question)
print(context_result)

[Document(metadata={'id': 'doc3', 'chunk_id': 'doc3_chunk0', 'distance': 0.28105175495147705}, page_content='Unsupervised learning involves training models on data that does not contain labels. The goal is to discover hidden patterns or structures within the dataset')]


In [ ]:
context_result

[Document(metadata={'id': 'doc3', 'chunk_id': 'doc3_chunk0', 'distance': 0.28105175495147705}, page_content='Unsupervised learning involves training models on data that does not contain labels. The goal is to discover hidden patterns or structures within the dataset')]

In [ ]:
context_result_question

[Document(metadata={'id': 'doc3', 'chunk_id': 'doc3_chunk0'}, page_content='Unsupervised learning involves training models on data that does not contain labels. The goal is to discover hidden patterns or structures within the dataset'),
 Document(metadata={'id': 'doc3', 'chunk_id': 'doc3_chunk1'}, page_content='. Techniques such as clustering and dimensionality reduction are commonly used. These methods are helpful for exploratory data analysis and feature discovery.'),
 Document(metadata={'id': 'doc5', 'chunk_id': 'doc5_chunk0'}, page_content='Overfitting occurs when a machine learning model learns the training data too well, including its noise and irrelevant details. As a result, the model performs poorly on new, unseen data'),
 Document(metadata={'id': 'doc5', 'chunk_id': 'doc5_chunk1'}, page_content='. Techniques such as regularization, dropout, and cross-validation help reduce overfitting. A good model balances learning patterns while maintaining generalization.'),
 Document(meta

In [ ]:
q_results

[(Document(metadata={'docs': '["doc3", "doc15", "doc3", "doc10", "doc5"]', 'id': 'q6'}, page_content='Which techniques are typically applied in unsupervised learning tasks?'),
  0.31184783577919006)]

In [ ]:
len(context_result_question)

8